In [24]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import cv2 as cv
import numpy as np
import time
import os
from collections import deque

In [26]:
# INPUT
vid_path = input("Enter the video path: ")
if vid_path == "0":
  cap = cv.VideoCapture(0) # I am not sure about this condition for live webcam feeds
else:
cap = cv.VideoCapture(video_path) # please check the output result for static video files

In [27]:
# OUTPUT VIDEO SETUP
fps = 30 # frame per seconds (min value = 15)
frame_w = 640
frame_h = 400

fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter("output.mp4", fourcc, fps, (frame_w, frame_h))

In [28]:
# Initial variables

prev_gray = None
centroid_history = deque(maxlen=10)
alarm_start = None
status = "SAFE"

In [29]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv.resize(frame, (frame_w, frame_h))


    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    gray = cv.GaussianBlur(gray, (5,5), 0)

    if prev_gray is None:
        prev_gray = gray
        continue


    # Frame differencing

    diff = cv.absdiff(prev_gray, gray)

    # Binary threshold
    _, thresh = cv.threshold(diff, 25, 255, cv.THRESH_BINARY)


    # Morphological Operations

    kernel = np.ones((3,3), np.uint8)

    thresh = cv.morphologyEx(thresh, cv.MORPH_OPEN, kernel, iterations=2)
    thresh = cv.dilate(thresh, kernel, iterations=2)
    thresh = cv.morphologyEx(thresh, cv.MORPH_CLOSE, kernel, iterations=2)

    frame_a = frame_w * frame_h

    # Motion Footprint

    motion_pixels = cv.countNonZero(thresh)
    motion_footprint = (motion_pixels / frame_a) * 100

    # Active zone visualization

    active_mask = cv.cvtColor(thresh, cv.COLOR_GRAY2BGR)
    active_mask[:, :, 0] = thresh
    frame = cv.addWeighted(frame, 0.8, active_mask, 0.5, 0)


    # Contour Detection

    contours, _ = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    if len(contours) > 0:
        cnt = max(contours, key=cv.contourArea)
        area = cv.contourArea(cnt)

        if area > 500:
            x, y, w, h = cv.boundingRect(cnt)


            # Centroid

            M = cv.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
            else:
                cx, cy = 0, 0

            centroid = (cx, cy)
            centroid_history.append(centroid)

            # velocity estimation

            velocity = 0
            if len(centroid_history) >= 2:
                dx = centroid_history[-1][0] - centroid_history[-2][0]
                dy = centroid_history[-1][1] - centroid_history[-2][1]
                velocity = np.sqrt(dx**2 + dy**2) * fps


            # Intruder ID

            mean_intensity = np.mean(gray[y:y+h, x:x+w])
            intruder_id = int(area + mean_intensity) % 10000

            # Locating the intruder

            cv.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 2)
            cv.putText(frame, "ACTIVE ZONE", (x, y+h+20),cv.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 2)

            cv.circle(frame, centroid, 5, (0,0,255), -1)

            if len(centroid_history) >= 2:
                cv.arrowedLine(frame,centroid_history[-2],centroid_history[-1],(255,0,0), 2)


            # Alarm logic

            if motion_footprint > 2:
                if alarm_start is None:
                    alarm_start = time.time()
                elif time.time() - alarm_start > 2:
                    status = "ALARM"
                    cv.putText(frame, "ALARM!", (20,50),
                                cv.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)
            else:
                alarm_start = None
                status = "SAFE"


            # Display of info

            cv.putText(frame, f"ID: {intruder_id}", (x,y-10),cv.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

            cv.putText(frame, f"Centroid: ({cx},{cy})", (cx+10, cy),cv.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)

            cv.putText(frame, f"Motion Footprint: {motion_footprint:.2f}%", (20,100),cv.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

            cv.putText(frame, f"Velocity: {velocity:.2f} px/s", (20,130),cv.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)


    # status display

    color = (0,255,0) if status == "SAFE" else (0,0,255)

    cv.putText(frame, f"Status: {status}", (20,160),cv.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    out.write(frame)
    prev_gray = gray

In [31]:
# OUTPUT
os.makedirs("frames", exist_ok=True)
from google.colab import files
files.download("output.mp4")

cap.release()
out.release()

print("✅ Final Output saved as output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Final Output saved as output.mp4
